# Exploración y selección de variables para el modelo

Este notebook es la entrada a la fase de modelado. Lee los Parquet anuales ya validados y responde cuatro preguntas: qué representa cada fila, cómo se distribuyen las igniciones, qué variables son candidatas y cuáles son redundantes. No descarga datos, no modifica el dataset y no entrena ningún modelo.

## Contrato experimental

- **Unidad de análisis:** una celda activa de 1 km de Galicia en una fecha con cobertura EGIF.
- **Target:** `target_ignicion`; vale 1 si EGIF registra una ignición en la celda y día, y 0 en caso contrario.
- **Partición que se usará más adelante:** entrenamiento 2019–2021, validación 2022 y test ciego 2023. Aquí se exploran 2019–2022; 2023 permanece fuera de las decisiones de selección.
- **Contrato meteorológico:** las variables describen el día T, sin desplazamiento temporal. En operación, para predecir T deberán sustituirse por la previsión meteorológica disponible para T; el target EGIF nunca es una entrada.

La muestra se crea al leer por lotes: conserva todas las igniciones y selecciona negativos con una regla determinista sobre `cell_id`. Por ello es reproducible, contiene eventos raros y no intenta cargar los Parquet completos en memoria.

In [ ]:
from collections import defaultdict
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.dataset as pads
import seaborn as sns
from sklearn.metrics import roc_auc_score

plt.style.use('default')
plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'savefig.facecolor': 'white', 'figure.dpi': 110,
})
sns.set_theme(style='whitegrid', context='notebook')

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
DATASET_DIR = ROOT / 'data' / 'processed' / 'tabular' / 'egif'
METADATA_PATH = DATASET_DIR / 'metadata.json'

assert METADATA_PATH.exists(), (
    'La exportación aún no ha terminado: falta metadata.json. '
    'Espera a que se consoliden todos los años antes de ejecutar este notebook.'
)
metadata = json.loads(METADATA_PATH.read_text(encoding='utf-8'))
annual_files = sorted(DATASET_DIR.glob('year=*/dataset_*.parquet'))
fragments = list(DATASET_DIR.glob('year=*/part-*.parquet')) + list(DATASET_DIR.glob('year=*/*.partial'))
assert len(annual_files) == 5, f'Se esperaban cinco Parquet anuales; encontrados: {annual_files}'
assert not fragments, f'La consolidación no ha terminado: {fragments[:3]}'

dataset = pads.dataset([str(path) for path in annual_files], format='parquet')
# Compatibilidad: versiones antiguas publicaban `year` erróneamente como predictor.
# Se mantiene en el Parquet para los cortes temporales, pero no entra en el modelo.
PREDICTORS = [name for name in metadata['predictor_columns'] if name != 'year']
TARGET = 'target_ignicion'
IDENTIFIERS = ['fecha', 'cell_id', 'year']
OUTCOMES = set(metadata['outcome_columns_not_predictors'])

print(f'Ruta: {DATASET_DIR}')
print(f'Filas exportadas: {sum(metadata["annual_files"].values()):,}')
if 'year' in metadata['predictor_columns']:
    print('Aviso: se excluye `year` del metadato heredado; no es un predictor válido.')
print(f'Predictores disponibles: {len(PREDICTORS)}')
print('Contrato temporal:', metadata['time_contract'])

## 1. Esquema y barreras contra *data leakage*

Las variables resultado no pueden ser predictores. En particular, `burned_area_ha` y `large_fire_500ha` se conocen después de la ignición. `fecha`, `cell_id` e `is_galicia` son identificadores o máscara espacial: tampoco entrarán en el primer modelo.

In [ ]:
schema = pd.DataFrame({
    'columna': dataset.schema.names,
    'tipo': [str(field.type) for field in dataset.schema],
})
display(schema)

forbidden = OUTCOMES | {'fecha', 'cell_id', 'year', 'is_galicia'}
leaks = sorted(set(PREDICTORS) & forbidden)
assert not leaks, f'Predictores prohibidos detectados: {leaks}'
assert TARGET not in PREDICTORS
print('✓ El esquema separa identificadores, resultados y predictores.')

## 2. Cobertura temporal y prevalencia real

Esta agregación recorre solo fecha y target por lotes. La tasa de ignición se calcula sobre todos los registros, no sobre una muestra balanceada. Es la referencia correcta para interpretar más adelante PR-AUC, precisión y calibración.

In [ ]:
coverage = defaultdict(lambda: [0, 0])
for batch in dataset.scanner(columns=['fecha', TARGET], batch_size=250_000).to_batches():
    frame = batch.to_pandas()
    dates = pd.to_datetime(frame['fecha'])
    grouped = pd.DataFrame({
        'year': dates.dt.year,
        'month': dates.dt.month,
        TARGET: frame[TARGET].to_numpy(),
    }).groupby(['year', 'month'])[TARGET].agg(['size', 'sum'])
    for (year, month), row in grouped.iterrows():
        coverage[(year, month)][0] += int(row['size'])
        coverage[(year, month)][1] += int(row['sum'])

monthly = pd.DataFrame(
    [(year, month, rows, positives) for (year, month), (rows, positives) in coverage.items()],
    columns=['year', 'month', 'filas', 'igniciones'],
).sort_values(['year', 'month'])
monthly['prevalencia_pct'] = 100 * monthly['igniciones'] / monthly['filas']
display(monthly.groupby('year')[['filas', 'igniciones']].sum().assign(
    prevalencia_pct=lambda x: 100 * x['igniciones'] / x['filas']
))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
monthly.groupby('year')['igniciones'].sum().plot.bar(ax=axes[0], color='tab:red')
axes[0].set(title='Igniciones EGIF por año', xlabel='Año', ylabel='n.º de celdas con ignición')
sns.lineplot(data=monthly, x='month', y='prevalencia_pct', hue='year', marker='o', ax=axes[1])
axes[1].set(title='Estacionalidad: prevalencia mensual real', xlabel='Mes', ylabel='% de filas con ignición')
plt.tight_layout()

assert monthly['filas'].gt(0).all()
print(f'Prevalencia global: {100 * monthly.igniciones.sum() / monthly.filas.sum():.4f}%')

## 3. Catálogo de conjuntos candidatos

No se elige una variable solo porque tenga una correlación alta en una muestra. Primero se compararán grupos con sentido físico: meteorología, calendario y contexto estático. La decisión final se hará en 2022 con un modelo tabular y se confirmará una sola vez en 2023.

In [ ]:
METEOROLOGY_FEATURES = [
    'temperature_mean', 'temperature_min', 'temperature_max', 'temperature_max_12_18h',
    'relative_humidity_mean', 'relative_humidity_min', 'relative_humidity_min_12_18h',
    'wind_speed_mean', 'wind_speed_max', 'wind_speed_max_12_18h',
    'precipitation_sum', 'precipitation_sum_3d',
    'precipitation_sum_7d', 'precipitation_sum_14d', 'precipitation_sum_30d',
    'temperature_mean_7d', 'relative_humidity_mean_7d', 'consecutive_dry_days',
]
CALENDAR_FEATURES = [
    'month', 'iso_week', 'day_of_year', 'day_of_week', 'is_weekend',
    'day_of_year_sin', 'day_of_year_cos', 'month_sin', 'month_cos',
]
TOPOGRAPHY_FEATURES = [name for name in PREDICTORS if name.startswith((
    'elevation_', 'slope_', 'roughness_', 'aspect_',
))]
LANDCOVER_NAMES = {
    'agriculture', 'artificial', 'broadleaf_forest', 'coniferous_forest',
    'mixed_forest', 'open_spaces', 'scrub', 'water', 'wetlands',
    'forest_cover_fraction', 'combustible_pct_forestal', 'combustible_clase',
}
LANDCOVER_FEATURES = [name for name in PREDICTORS if name in LANDCOVER_NAMES]

FEATURE_SETS = {
    'A_meteorologia': METEOROLOGY_FEATURES,
    'B_meteo_calendario': METEOROLOGY_FEATURES + CALENDAR_FEATURES,
    'C_contexto_ambiental': METEOROLOGY_FEATURES + CALENDAR_FEATURES + TOPOGRAPHY_FEATURES + LANDCOVER_FEATURES,
}
for name, features in FEATURE_SETS.items():
    missing = sorted(set(features) - set(PREDICTORS))
    assert not missing, f'{name}: columnas ausentes {missing}'

catalogue = pd.DataFrame({
    'conjunto': FEATURE_SETS.keys(),
    'n_predictores': [len(features) for features in FEATURE_SETS.values()],
    'variables': [', '.join(features) for features in FEATURE_SETS.values()],
})
display(catalogue)
assert set(FEATURE_SETS['C_contexto_ambiental']) == set(PREDICTORS)
print('✓ Los conjuntos A, B y C cubren exactamente los predictores disponibles.')

## 4. Muestra reproducible para exploración

La exploración profunda usa los años 2019–2022. Se conservan todas las igniciones y aproximadamente una de cada 251 celdas negativas. Esto conserva la estructura temporal, evita agotar la memoria y deja 2023 sin usar. La primera ejecución lee los cuatro Parquet de entrenamiento/validación y puede tardar unos minutos; después la muestra queda solo en memoria.

Para una prueba rápida cambia `EDA_YEARS` a `(2022,)`. No modifiques el módulo sin anotarlo en los resultados.

In [ ]:
EDA_YEARS = (2019, 2020, 2021, 2022)
NEGATIVE_CELL_MODULUS = 251
sample_files = [
    path for path in annual_files
    if int(path.parent.name.removeprefix('year=')) in EDA_YEARS
]
sample_dataset = pads.dataset([str(path) for path in sample_files], format='parquet')
sample_columns = IDENTIFIERS + [TARGET] + PREDICTORS
sample_frames = []
for batch in sample_dataset.scanner(columns=sample_columns, batch_size=100_000).to_batches():
    frame = batch.to_pandas()
    keep = (frame[TARGET].to_numpy() == 1) | (frame['cell_id'].to_numpy() % NEGATIVE_CELL_MODULUS == 0)
    if keep.any():
        sample_frames.append(frame.loc[keep])

sample = pd.concat(sample_frames, ignore_index=True)
sample['fecha'] = pd.to_datetime(sample['fecha'])
sample['year'] = sample['fecha'].dt.year
print(f'Muestra: {len(sample):,} filas; igniciones: {sample[TARGET].sum():,}; prevalencia: {100 * sample[TARGET].mean():.3f}%')
print('Nota: esta prevalencia está enriquecida por las igniciones; la prevalencia real es la de la sección 2.')
assert sample['year'].isin(EDA_YEARS).all()
assert sample.loc[sample[TARGET] == 1].shape[0] == monthly.loc[monthly.year.isin(EDA_YEARS), 'igniciones'].sum()

## 5. Calidad y distribución de las variables

El notebook de validación Parquet ya verifica la integridad completa. Esta sección describe la muestra que se utilizará para análisis visual: tasas de ausencia, percentiles y rangos. Una variable con ausencia, rango o unidad inesperados se corrige en la fase de datos, no mediante imputación silenciosa durante el entrenamiento.

In [ ]:
numeric_features = sample[PREDICTORS].select_dtypes(include=np.number).columns.tolist()
quality = sample[numeric_features].describe(percentiles=[.01, .5, .99]).T
quality['missing_pct'] = 100 * sample[numeric_features].isna().mean()
quality['n_unique'] = sample[numeric_features].nunique()
display(quality.sort_values('missing_pct', ascending=False))

assert quality['missing_pct'].max() == 0, 'Hay predictores ausentes: revisar el pipeline antes de entrenar.'
assert sample['relative_humidity_min_12_18h'].between(0, 100).all()
assert (sample[['wind_speed_max_12_18h', 'precipitation_sum', 'consecutive_dry_days']] >= 0).all().all()
print('✓ La muestra no contiene nulos ni rangos meteorológicos imposibles.')

## 6. Señal univariante

Un AUC univariante mide cuánto ordena una sola variable a las igniciones en esta muestra; se muestra en valor absoluto para no penalizar una relación inversa, por ejemplo más precipitación implica menor riesgo. No sirve para decidir por sí solo: variables correlacionadas, no linealidades e interacciones se evaluarán con el modelo temporal de validación.

In [ ]:
univariate_rows = []
for feature in numeric_features:
    values = sample[feature]
    if values.nunique() < 2:
        continue
    auc = roc_auc_score(sample[TARGET], values)
    univariate_rows.append({
        'variable': feature,
        'auc': auc,
        'fuerza_ordenacion': max(auc, 1 - auc),
        'direccion': 'directa' if auc >= .5 else 'inversa',
    })
univariate = pd.DataFrame(univariate_rows).sort_values('fuerza_ordenacion', ascending=False)
display(univariate)

top_features = univariate.head(12).sort_values('fuerza_ordenacion')
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(top_features['variable'], top_features['fuerza_ordenacion'], color='tab:blue')
ax.axvline(.5, color='black', linestyle='--', linewidth=1)
ax.set(title='Señal univariante en la muestra', xlabel='max(AUC, 1 − AUC)', ylabel='')
plt.show()

display(sample.groupby(TARGET)[top_features['variable'].tail(4).tolist()].median().T)

## 7. Redundancia entre predictores

Las ventanas de precipitación, las temperaturas y las fracciones de CORINE pueden estar muy correlacionadas. Los árboles toleran cierta correlación, pero mantener pares casi idénticos aumenta complejidad y hace inestable la importancia. Esta tabla propone pares para revisar; no los elimina automáticamente.

In [ ]:
correlation = sample[numeric_features].corr(method='spearman')
upper = correlation.where(np.triu(np.ones(correlation.shape), k=1).astype(bool))
high_pairs = upper.stack().rename('spearman').reset_index()
high_pairs.columns = ['variable_1', 'variable_2', 'spearman']
high_pairs = (
    high_pairs.loc[high_pairs['spearman'].abs() >= .90]
    .sort_values('spearman', key=lambda x: x.abs(), ascending=False)
    .reset_index(drop=True)
)
display(high_pairs)

focus = univariate.head(15)['variable'].tolist()
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(correlation.loc[focus, focus], cmap='vlag', center=0, square=True, ax=ax)
ax.set_title('Correlación de Spearman: 15 variables con mayor señal univariante')
plt.tight_layout()

print('Revisar especialmente las ventanas acumuladas; su correlación es esperable y tiene interpretación física.')

## 8. Decisión antes de entrenar

Al terminar, registra en la memoria del TFM: prevalencia real por año, variables con problemas, pares redundantes revisados y los tres conjuntos A/B/C. No uses todavía métricas de 2023.

El siguiente notebook o script entrenará primero un modelo tabular en 2019–2021, elegirá el conjunto y los hiperparámetros exclusivamente en 2022, y solo entonces hará una evaluación final en 2023. La importancia por permutación y SHAP se calcularán sobre esa validación, no sobre el test.